# Exercise 1: Scrambling and case

*Syntax and Universal Dependencies · version 0.1.0*

> Write a Python function that takes a Turkish sentence in CoNLL-U format, identifies the subject (nsubj), object (obj), and root verb, and reports their relative positions (subject-before-verb? object-before-verb?). Apply it to the first 100 sentences of the TR-BOUN training split that contain an overt subject and an overt object of the root (report how many sentences you had to skip). What percentage of sentences have canonical SOV order? What order variants appear, and how often?

**Setup.** `pip install "turkicnlp==0.5.0" "stanza==1.11.0" scipy matplotlib requests`. Stanza is pinned to 1.11.0 because the worked solutions use its Turkish (IMST), Kazakh (KTB) and Kyrgyz (KTMU) checkpoints; with another version, Stanza downloads that version's models and the numbers can change. Models already on disk are used from `STANZA_RESOURCES_DIR` (default `~/stanza_resources`); missing ones are downloaded. The Universal Dependencies files are the **UD 2.17** release files (tag `r2.17` on GitHub). They are downloaded once into `UD217_DIR` (default `~/.cache/ud217`) and checked against the SHA-256 hashes recorded in the book's treebank inventory. Parsers receive the **gold words** (`tokenize_pretokenized=True`), so tokenisation errors do not enter the scores.

**Definitions.** A sentence is used if its root has at least one dependent labelled `nsubj` (or a subtype) and one labelled `obj`. Each argument is placed at the position of its head word, so a long subject counts by its head noun. The predicate position is the root, whatever its UPOS: under the BOUN convention a complex predicate such as *rahatsız et-* has the nominal as root and the light verb as its dependent (Syntax and Universal Dependencies, light-verb table), so some roots are nouns or adjectives. The same function is then applied to the whole BOUN and IMST training splits.

In [ ]:
import collections, hashlib, math, os, random, statistics
from pathlib import Path
import requests

# Pinned UD 2.17 release files (the inventory in tables/tbl-ud217-inventory) and their SHA-256 hashes.
UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/r2.17/{file}"
SHA256 = {
    "UD_Turkish-BOUN/tr_boun-ud-train.conllu": "373fe39e3acdba3219521f54fa5ae768851fbb4bf9a31079ad1ca6590410650c",
    "UD_Turkish-BOUN/tr_boun-ud-test.conllu": "4f74fc0f9efdd1285d781f3efd1e98cbf39cf4433df9eb4dc14162e8ed36773b",
    "UD_Turkish-IMST/tr_imst-ud-train.conllu": "8d8b8b581dab9362e6f2e4328aef77b9fabec1d189a199e320ba136710c7bd83",
    "UD_Turkish-IMST/tr_imst-ud-test.conllu": "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
    "UD_Turkish-TueCL/tr_tuecl-ud-test.conllu": "2508f47e57b9b6a1d7241d0ab4a918d84c5cbb0455bbd961ac5231e31a1d311e",
    "UD_Azerbaijani-TueCL/az_tuecl-ud-test.conllu": "8b315f8b065f094e38482465706ba44a9caf1b082fc8bbc94908f6fdeee8c3b5",
    "UD_Kyrgyz-TueCL/ky_tuecl-ud-test.conllu": "08224be6b74346f582234f54527412dd6265c8d8b42e7d6ef463aade85bdcd9b",
    "UD_Kazakh-KTB/kk_ktb-ud-test.conllu": "1100846ee50c2f970562a21dca8415a381c470ea06bdb8496ae213d59c3a3c4e",
    "UD_English-EWT/en_ewt-ud-train.conllu": "6fe5692f2e02198262483daba0626efe346dd19f0ce6c3ee7728b08af9b90a9c",
}
CACHE = Path(os.environ.get("UD217_DIR", "~/.cache/ud217")).expanduser()

def ud_path(key):
    """Local copy of a pinned UD 2.17 file, downloaded once and checked against its SHA-256 hash."""
    path = CACHE / key
    if not path.exists():
        repo, file = key.split("/")
        r = requests.get(UD.format(repo=repo, file=file), timeout=120)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != SHA256[key]:
        raise ValueError(f"{key} differs from the UD 2.17 release file")
    return path

Word = collections.namedtuple("Word", "id form lemma upos xpos feats head deprel")

def read(key_or_path):
    """Sentences as dicts: sent_id, text and words (integer-ID syntactic words only;
    multiword-token ranges and empty nodes are skipped)."""
    path = ud_path(key_or_path) if key_or_path in SHA256 else Path(key_or_path)
    sents, meta, words = [], {}, []
    for line in path.read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words:
                sents.append({"sent_id": meta.get("sent_id"), "text": meta.get("text"), "words": words})
            meta, words = {}, []
        elif line.startswith("#"):
            k, _, v = line[1:].partition("=")
            meta.setdefault(k.strip(), v.strip())
        else:
            c = line.split("\t")
            if c[0].isdigit():
                words.append(Word(int(c[0]), c[1], c[2], c[3], c[4], c[5], int(c[6]), c[7]))
    return sents

def feats(w):
    return dict(f.split("=", 1) for f in w.feats.split("|") if "=" in f)

def children(words):
    out = collections.defaultdict(list)
    for w in words:
        out[w.head].append(w)
    return out

def subtree(words, i, kids=None):
    """IDs of word i and all its descendants."""
    kids = kids or children(words)
    out, stack = set(), [i]
    while stack:
        j = stack.pop(); out.add(j); stack.extend(k.id for k in kids[j])
    return out

def q(values, p):
    """Nearest-rank quantile, as in the chapter's arc-length table."""
    v = sorted(values)
    return v[math.ceil(len(v) * p) - 1] if v else None

def stanza_pipeline(lang, processors="tokenize,pos,lemma,depparse"):
    """A Stanza pipeline over gold words (tokenize_pretokenized=True). Uses models already on disk
    (STANZA_RESOURCES_DIR or ~/stanza_resources) and downloads them only if they are missing."""
    import logging, stanza
    logging.getLogger("stanza").setLevel(logging.ERROR)
    model_dir = os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser()))
    have = (Path(model_dir) / lang / "depparse").exists()
    return stanza.Pipeline(lang, processors=processors, tokenize_pretokenized=True, model_dir=model_dir,
                           download_method=None if have else "reuse_resources", verbose=False)

def parse_gold_words(nlp, sents, forms=None):
    """Parse each gold sentence from its gold words (optionally replaced by `forms`, one list per
    sentence, e.g. transliterated) and return predicted sentences with the original word forms."""
    forms = forms or [[w.form for w in s["words"]] for s in sents]
    doc = nlp(forms)
    out = []
    for s, ps in zip(sents, doc.sentences):
        assert len(ps.words) == len(s["words"]), s["sent_id"]
        out.append({"sent_id": s["sent_id"], "text": s["text"], "words": [
            Word(g.id, g.form, p.lemma or "_", p.upos or "_", p.xpos or "_", p.feats or "_", p.head, p.deprel)
            for g, p in zip(s["words"], ps.words)]})
    return out

def write_conllu(sents, path):
    with open(path, "w", encoding="utf-8") as f:
        for s in sents:
            f.write(f"# sent_id = {s['sent_id']}\n# text = {s['text']}\n")
            for w in s["words"]:
                f.write("\t".join([str(w.id), w.form, w.lemma or "_", w.upos, w.xpos or "_", w.feats or "_",
                                   str(w.head), w.deprel, "_", "_"]) + "\n")
            f.write("\n")

def score(gold, pred):
    """UPOS, UAS and LAS (%) with the official CoNLL 2018 scorer, over gold-word input.
    Gold multiword tokens are not written, so both files have the same words and text."""
    import tempfile
    from stanza.utils import conll18_ud_eval as ev
    with tempfile.TemporaryDirectory() as tmp:
        g, p = Path(tmp) / "gold.conllu", Path(tmp) / "pred.conllu"
        write_conllu(gold, g); write_conllu(pred, p)
        r = ev.evaluate(ev.load_conllu_file(str(g)), ev.load_conllu_file(str(p)))
    return {k: round(100 * r[k].f1, 2) for k in ("Words", "UPOS", "UAS", "LAS")}

In [ ]:
def order(s):
    """Order of subject, object and root predicate, or None if the root lacks an overt nsubj or obj.
    Each argument is placed at the position of its head word."""
    ws = s["words"]
    root = next(w for w in ws if w.head == 0)
    deps = [w for w in ws if w.head == root.id]
    subj = [w for w in deps if w.deprel.split(":")[0] == "nsubj"]
    obj = [w for w in deps if w.deprel == "obj"]
    if not subj or not obj:
        return None
    pos = sorted([(subj[0].id, "S"), (obj[0].id, "O"), (root.id, "V")])
    return "".join(x for _, x in pos), root.upos

def orders(key, limit=None):
    seen = skipped = 0; counts = collections.Counter(); root_upos = collections.Counter()
    for s in read(key):
        o = order(s)
        if o is None:
            skipped += 1; continue
        counts[o[0]] += 1; root_upos[o[1]] += 1; seen += 1
        if limit and seen == limit:
            break
    return counts, root_upos, skipped

for key, limit in [("UD_Turkish-BOUN/tr_boun-ud-train.conllu", 100),
                   ("UD_Turkish-BOUN/tr_boun-ud-train.conllu", None),
                   ("UD_Turkish-IMST/tr_imst-ud-train.conllu", None)]:
    counts, root_upos, skipped = orders(key, limit)
    n = sum(counts.values())
    label = key.split("/")[1] + (f", first {limit}" if limit else ", all")
    print(f"{label}: {n} sentences used, {skipped} skipped; root UPOS {dict(root_upos.most_common(3))}")
    print("   " + ", ".join(f"{o} {c} ({c / n:.1%})" for o, c in counts.most_common()))

# Marked orders among the first 100 BOUN sentences, with the root's subject and object.
seen = 0
for s in read("UD_Turkish-BOUN/tr_boun-ud-train.conllu"):
    o = order(s)
    if o is None:
        continue
    seen += 1
    if o[0] != "SOV":
        ws = s["words"]; root = next(w for w in ws if w.head == 0)
        subj = next(w for w in ws if w.head == root.id and w.deprel.split(":")[0] == "nsubj")
        obj = next(w for w in ws if w.head == root.id and w.deprel == "obj")
        print(f"   {o[0]} {s['sent_id']:9} S={subj.form} O={obj.form} V={root.form}  | {s['text'][:70]}")
    if seen == 100:
        break

## Worked solution

Output of `experiments/ch08_exercises/run_ch08_exercises.py` (UD 2.17 release files):

```
tr_boun-ud-train.conllu, first 100: 100 sentences used, 776 skipped; root UPOS {'VERB': 81, 'NOUN': 14, 'ADJ': 5}
   SOV 86 (86.0%), OSV 8 (8.0%), SVO 5 (5.0%), OVS 1 (1.0%)
tr_boun-ud-train.conllu, all: 957 sentences used, 6846 skipped; root UPOS {'VERB': 827, 'NOUN': 104, 'ADJ': 19}
   SOV 834 (87.1%), OSV 64 (6.7%), OVS 32 (3.3%), SVO 24 (2.5%), VSO 2 (0.2%), VOS 1 (0.1%)
tr_imst-ud-train.conllu, all: 487 sentences used, 2948 skipped; root UPOS {'VERB': 421, 'NOUN': 38, 'ADJ': 14}
   SOV 428 (87.9%), OSV 29 (6.0%), SVO 19 (3.9%), OVS 10 (2.1%), VOS 1 (0.2%)
   OSV bio_1512  S=o O=Bunu V=yapmış  | "Bunu da mı o yapmış?"
   OSV ess_1231  S=hakkı O=muhalefete V=tanınmamıştır  | 1925'in ilk yarısından itibaren, Türkiye'de muhalefete ve dolayısıyla 
   SVO bio_213   S=Yayıncılar O=birbirlerini V=biliyorlar  | Yayıncılar, editörler, şaşılacak şey, röntgen gibi biliyorlar birbirle
   SVO bio_179   S=biz O=yemeğimize V=başladık  | Yan odadaki biz gazeteciler, baş konuktan ya iki ya da en fazla üç dak
   SVO pop_1870  S=sözcüğü O=kavramı V=dönüştürür  | İntihar, başlı başına umutsuzluğu ifade ederken, yanına eklenen "saldı
   OSV bio_1101  S=hareketler O=ritmi V=yapıyor  | İleri geri ritmi hareketler yapıyor.
   OSV ess_1565  S=kendi O=Putunu V=yapar  | Putunu kendi yapar kendi tapar. " Ben kendimle konuşuyorum ama kimseyl
   OSV bio_135   S=kendileri O=başkaları V=almış  | Böylece başkaları yaptığında "taraflı, şartlı" demelerin önünü kendile
   SVO pop_896   S=Alfred O=Barış V=izlemiş  | Programın yapımcısı Alfred Biolek, bir konserinde izlemişti Barış Manç
   OSV bio_1899  S=siz O=kızımı V=üzün  | Benim kızımı siz böyle üzün, ha? "Bırak Allahaşkına ağlamayı" diyorum,
   OSV ess_1826  S=şeyler O=onu V=rahatsız  | Çünkü onu daima bir şeyler çok rahatsız ediyor...
   OVS bio_169   S=Zeffirelli O=sahnelerini V=çalıştırmış  | Bir de düello sahnelerini çok çalıştırmış Zeffirelli.
   OSV news_1810 S=Adnanlar O=Aslan'ı V=yaktı  | Aslan'ı Adnanlar yaktı
   SVO news_1884 S=Biz O=insanları V=gördük  | Biz daha önce de gördük bu insanları.
```

**The first 100 sentences.** To find 100 sentences whose root has both an overt subject and an overt object, 776 BOUN training sentences had to be skipped; most sentences lack one of the two (Exercise 3 measures how often subjects are omitted). Of the 100, 86 are SOV, 8 OSV, 5 SVO and 1 OVS; no verb-initial order occurs.

**The whole split.** Over all 957 qualifying BOUN training sentences the proportions are almost the same: 87.1% SOV, then OSV (6.7%), OVS (3.3%) and SVO (2.5%), and only three verb-initial sentences (two VSO, one VOS). IMST gives 87.9% SOV, with the same ranking except that SVO is ahead of OVS. All six orders are possible, but in edited text the marked orders are rare, and verb-initial orders almost absent. A parser that has learned "the nominal before the verb is the subject or object" is right most of the time; the case suffix is what tells it which is which, and the remaining eighth of the sentences is where position alone fails.

**What the examples show.** Several OSV examples front a pronoun or a topical object (*Bunu da mı o yapmış?* 'Did he do *this* too?', *Aslan'ı Adnanlar yaktı*), the postverbal constituents are afterthoughts (*Biz daha önce de gördük bu insanları*), and OVS puts the subject after the verb (*… çok çalıştırmış Zeffirelli*), as the chapter describes. Two caveats come from the annotation, not the language: BOUN labels some dative complements `obj` (*yemeğimize başladık*), and with nominal roots (*rahatsız* in *rahatsız ediyor*) the "V" position is the nominal part of the predicate. Counting by the position of the head word also hides how much material precedes or follows it.

## Your turn — extension

1. Place each argument at the first word of its subtree instead of its head word. Do any orders change?
2. Restrict `obj` to accusative-marked objects (`Case=Acc`) and compare the order distribution of definite and indefinite objects.